In [1]:
from arango import ArangoClient, ServerVersionError
from datetime import datetime, timedelta
from ollama import Client

import getpass
import json
import os
import random
import threading
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from multiprocessing import Lock
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

## Functions
##### Run Cells to Continue

In [2]:
def connect_to_arango_client(host: str):
    try:
        client = ArangoClient(hosts=host)
    except:
        print(f'{get_timestamp()} -- Could not connect to ArangoDB client at "{host}"')
        return None
    
    return client

In [3]:
def connect_to_arango_db(client: ArangoClient, db_name: str, username: str, password: str, max_retries: int = 3): 
    retries = 0
    if password is None:
        password = getpass.getpass(f'Please enter the password for user {username} for access to the {db_name} database: ')
        
    while retries < max_retries:
        try:
            db = client.db(db_name, username=username, password=password)     
            vers = db.version()
            print(f'{get_timestamp()} -- Successfully connected to database {db_name} running version {vers}.')
            return db
        except (ConnectionAbortedError, ServerVersionError) as e:
            retries += 1
            print(f'{get_timestamp()} -- Connection refused for db {db_name}.')
            password = getpass.getpass(f'Input password to try again:  ')
        except Exception as e:
            print(f'\n**** {type(e)}: {e} ****\n')
            retries += 1
            print(f'{get_timestamp()} -- Could not connect to db "{db_name}".')
    
    return None

In [4]:
def without(d, keys):
    new_d = d.copy()
    for key in keys:
        if key in d.keys():
            new_d.pop(key)
    return new_d

In [5]:
def get_timestamp(str_format: str = "%Y-%m-%d %H:%M:%S"):
    return datetime.now().strftime(str_format)

def connect_to_arango_client(host: str):
    try:
        client = ArangoClient(hosts=host)
    except:
        print(f'{get_timestamp()} -- Could not connect to ArangoDB client at "{host}"')
        return None
    
    return client


def connect_to_arango_db(client: ArangoClient, db_name: str, username: str, password: str, max_retries: int = 3): 
    retries = 0
    if password is None:
        password = getpass.getpass(f'Please enter the password for user {username} for access to the {db_name} database: ')
        
    while retries < max_retries:
        try:
            db = client.db(db_name, username=username, password=password)     
            vers = db.version()
            print(f'{get_timestamp()} -- Successfully connected to database {db_name} running version {vers}.')
            return db
        except (ConnectionAbortedError, ServerVersionError) as e:
            retries += 1
            print(f'{get_timestamp()} -- Connection refused for db {db_name}.')
            password = getpass.getpass(f'Input password to try again:  ')
        except Exception as e:
            print(f'\n**** {type(e)}: {e} ****\n')
            retries += 1
            print(f'{get_timestamp()} -- Could not connect to db "{db_name}".')
    
    return None

In [6]:
def exec_aql_query(aql, query):
    cursor = aql.execute(query)
    return cursor

In [7]:
def get_steps_for_all_ttps(aql, graph_name):
    ttp_steps_query = """
        LET step_collections = ["DevelopmentStep", "PlanningStep", "ExecutionStep"]

            FOR ttp IN TTPArtifact
                LET refs = (
                    FOR a IN INBOUND ttp GRAPH '""" + graph_name + """'    // *Artifact → TTPArtifact
                        FOR s IN INBOUND a GRAPH '""" + graph_name + """'   // *Step → *Artifact
                            LET step_type = SPLIT(s._id, "/")[0]
                            FILTER step_type IN step_collections
                            COLLECT stepType = step_type INTO grouped_artifacts = a
                            RETURN {
                                step_type: stepType,
                                reference_count: LENGTH(grouped_artifacts),
                                artifact_keys: UNIQUE(FOR g IN grouped_artifacts RETURN g._key)
                            }
                )
                RETURN {
                    ttp_key: ttp._key,
                    references_by_step_type: refs
                }
    """
    
    cursor = aql_query_graph(aql, ttp_steps_query)
    
    ttp_steps = [doc for doc in cursor]

    return ttp_steps


In [8]:
def get_sample_pairs_from_graph(db, aql, graph_name, 
                                n_samples=5, 
                                pairs_per_sample=5, 
                                src_filter_str='', 
                                pairs_filter_str='', 
                                src_collections=[], 
                                pairs_collections=[],
                                exclude_src_collections=[],
                                exclude_pairs_collections=[]
                               ):
    graph = db.graph(graph_name)

    # Get all vertex collections in the graph
    vertex_collections = graph.vertex_collections()
    if len(src_collections) == 0:
        src_collections = vertex_collections
    if len(pairs_collections) == 0:
        pairs_collections = vertex_collections
    
    src_included_collections = [coll for coll in vertex_collections if coll in src_collections and coll not in exclude_src_collections] 
    pairs_included_collections = [coll for coll in vertex_collections if coll in pairs_collections and coll not in exclude_pairs_collections]

    
    # Sample n nodes from each collection
    sampled_nodes = []
    limit_str = f'LIMIT {n_samples}' if n_samples > 0 else ''
    src_filter_str = f'FILTER {src_filter_str}' if src_filter_str != '' else ''
    pairs_filter_str = f'FILTER {pairs_filter_str}' if pairs_filter_str != '' else ''
    
    for vc in src_included_collections:
        query = f"""
        FOR doc IN {vc}
            {src_filter_str}
            SORT RAND()
            {limit_str}
            RETURN doc
        """
        result = list(aql.execute(query, batch_size=1000))
        #print(f'**** Len results: {len(result)} ****')
        sampled_nodes.extend([without(doc, ['_rev']) for doc in result])


    # Fetch all node documents (to choose random m for pairing) ---
    # Build a union query to get all documents from all vertex collections
    union_parts = [f"FOR doc IN {vc} {pairs_filter_str} RETURN doc" for vc in pairs_included_collections]
    all_docs_query = f"RETURN UNION({', '.join(union_parts)})" if len(pairs_included_collections) > 1 else union_parts[0]
    #print(f'all_docs_query:\n{all_docs_query}')

    cursor = list(aql.execute(all_docs_query, batch_size=10000))
    #print(isinstance(list(cursor)[0], dict))
    all_docs = [without(doc, ['_rev']) for batch in cursor for doc in batch] if len(pairs_included_collections) > 1 else [without(doc, ['_rev']) for doc in cursor]
    #print(all_docs[0])
    all_docs_by_id = {doc["_id"]: doc for doc in all_docs}
    
    # Build pairings
    pairings = []
    
    for node in sampled_nodes:
        node_id = node["_id"]
        
        # Exclude the current node from the pool
        candidates = [doc for doc_id, doc in all_docs_by_id.items() if doc_id != node_id]
        
        # Randomly sample m candidates
        if len(candidates) < pairs_per_sample or pairs_per_sample < 1:
            sampled = candidates
        else:
            sampled = random.sample(candidates, pairs_per_sample)

        #print(f'*** len(pairs): {len(sampled)} ***')
        pairings.append({'src_node': node, 'pair_nodes': sampled})

    return pairings


In [9]:
def get_graph_edges(db, aql, graph_name, include_node_docs=False):
    graph = db.graph(graph_name)
    valid_edges = []
    # Loop through edge definitions in the graph
    for ed in graph.edge_definitions():
        edge_collection = ed["edge_collection"]
        #print(ed)
        from_colls = ed["from_vertex_collections"]
        to_colls = ed["to_vertex_collections"]
    
        # Build AQL query for this edge definition
        # Accept any valid (from, to) pair from the defined collections
        filters = []
        for from_coll in from_colls:
            for to_coll in to_colls:
                filters.append(
                    f"(IS_SAME_COLLECTION(edge._from, '{from_coll}') && IS_SAME_COLLECTION(edge._to, '{to_coll}'))"
                )
        
        filter_clause = " || ".join(filters)
    
        aql_query = f"""
        FOR edge IN {edge_collection}
            FILTER {filter_clause}
            RETURN edge
        """
    
        result = list(aql.execute(aql_query, batch_size=1000))
        if include_node_docs:
            for edge in result:
                #print(edge)
                edge['_from_node'] = db.collection(edge['_from'].split('/')[0]).get(edge['_from'])
                edge['_to_node'] = db.collection(edge['_to'].split('/')[0]).get(edge['_to'])
                edge['type'] = edge_collection
            result = [edge for edge in result if edge['_from_node'] is not None and edge['_to_node'] is not None]
            
        valid_edges.extend(result)
    return valid_edges

In [10]:
def update_edgecoll_fields(db, aql, rename_fields={}, collections=[]):
    # --- Get all edge collections in the DB ---
    all_collections = db.collections()
    #rint(all_collections)
    edge_collections = [c["name"] for c in all_collections if c["type"] == 'edge'] if len(collections) == 0 else collections
    print(f"Found {len(edge_collections)} edge collections in the database.")
    
    # Build the merge dict part: { new_field: doc.old_field, ... }
    merge_fields = ", ".join(
        f"{new_field}: doc.{old_field}" for old_field, new_field in rename_fields.items()
    )

    # Build list of old fields to unset
    old_fields_list = ", ".join(f"'{old_field}'" for old_field in rename_fields.keys())
    
        # --- Run update on each edge collection ---
    for ec in edge_collections:
        print(f"Updating edge collection: {ec}")
        aql_query = f"""
        FOR doc IN {ec}
            FILTER {" || ".join(f"HAS(doc, '{old_field}')" for old_field in rename_fields.keys())}
            LET updated = UNSET(MERGE(doc, {{
                {merge_fields}
            }}), [{old_fields_list}])
            UPDATE doc WITH updated IN {ec}
        """
        print(aql_query)
        aql.execute(aql_query)
        print(f"Done updating {ec}")
    
    print("All edge documents updated.")

In [11]:
def show_n_ttps(ttps: list, n: int = 5, sort_on: str = None, from_end: str = 'top'):
    if from_end == 'top':
        if sort_on is not None:
            ttps = sorted(ttps, key=lambda x: x[sort_on], reverse=True)
        ttp_slice = ttps[:n]
        header_str_end = 'Top'
    elif from_end == 'bot':
        if sort_on is not None:
            ttps = sorted(ttps, key=lambda x: x[sort_on], reverse=False)
        ttp_slice = ttps[-n:]
        header_str_end = 'Bottom'
    else:
        print(f'Invalid value for from_end: {from_end}. Expected "top" or "bot".')

    header_str = f'********** {header_str_end} {n} TTPs **********'
    h_str_len = len(header_str)
    print(header_str)
    print('-'*h_str_len)
    print(f'   {"TTP":<13}|  {"References":<10}')
    print('-'*h_str_len)
    for ttp in ttp_slice:
        print(f'   {ttp["ttp_key"]:<13}|  {ttp["refs"]:>9}')

In [12]:
def plot_ttp_artifact_histogram(data: list[dict], save_to_file: bool = False, save_file: str = 'ttp_hist.png'):
    # Extract all unique step_types across all entries
    step_types = sorted({
        step['step_type']
        for entry in data
        for step in entry.get('references_by_step_type', [])
    })

    ttp_keys = [entry['ttp_key'] for entry in data]
    n = len(ttp_keys)
    x = np.arange(n)  # x positions for each ttp_key

    width = 0.5 / max(len(step_types), 1)  # dynamic bar width based on number of step_types

    # Prepare counts per step_type aligned with ttp_keys
    counts_per_step = {step: [] for step in step_types}
    total_counts = []
    for entry in data:
        step_count_map = {step['step_type']: step['reference_count'] for step in entry.get('references_by_step_type', [])}
        total = 0
        for step in step_types:
            count = step_count_map.get(step, 0)
            counts_per_step[step].append(step_count_map.get(step, 0))
            total += count
        total_counts.append(total)

    fig, ax = plt.subplots(figsize=(max(8, n), 5))

    # Plot bars for each step_type with proper offsets
    for i, step in enumerate(step_types):
        offset = (i - (len(step_types) - 1) / 2) * width
        ax.bar(x + offset, counts_per_step[step], width, label=step)

    # total counts bar
    ax.plot(x, total_counts, color='orchid', label='Total References', linewidth=0.7, linestyle='--')

    
    # Formatting
    ax.set_xlabel('TTP')
    ax.set_ylabel('Reference Count')
    ax.set_title('Reference Counts per Step Type for each TTP and Total per TTP')
    ax.set_xticks(x)
    ax.set_xticklabels(ttp_keys, rotation=45, ha='right')
    ax.legend(title="Step Type")
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    if save_to_file:
        fig.savefig(save_file)
    plt.show()

    


In [13]:
def almost_in_list(truth_list, edge, key, possible_maps={'suggested_vals': [], 'truth_vals': []}):
    if edge in truth_list:
        return True
    for sval in possible_maps['suggested_vals']:
        for tval in possible_maps['truth_vals']:
            edge_copy = edge.copy()
            edge_copy[key] = tval if edge[key] == sval else edge[key]
            if edge_copy in truth_list:
                return True
    return False

In [14]:
def prompt_and_response(client, model, prompt, sys_set=None, quiet=False, stream=True, options={}):
    p_start = datetime.now()
    response = ''
    try:
        for part in client.generate(model=model, prompt=prompt, stream=stream, options=options):
            response += part['response']
            if not quiet:
                print(part['response'], end='', flush=True)
    except Exception as e:
        print(f'ERROR: {e}\nSkipping prompt...')
        return None
    p_end = datetime.now() - p_start
    if not quiet:
        print(f'Prompt took {p_end}')
    return response

In [15]:
def part1_prompt(edge_pairs, oll_client, model='gemma3:27b-it-qat', quiet=True, options={'temperature': 0.8}, src_node_key='src_node', pair_node_key='pair_node'):
    part1_responses = []
    max_pairs_listed = 10
    prompts_start = datetime.now()
    edge_pairs
    pairings = [
        {
            'src_node': edge[src_node_key], 
            'pair_node': edge[pair_node_key]
        } for edge in edge_pairs if edge[src_node_key] is not None and edge[pair_node_key] is not None
    ]
    
    for node_pair in tqdm(pairings, desc='Prompt 1', leave=False):
        #print(node_pair)
        #print(node_pair["src_node"]["_id"])
        
        prompt = f"""Act as a data analyzer. Consider the following data for a graph network, as a source node and a destination node:
        {json.dumps(node_pair)}
        
        Should an edge exist between these two nodes?
        Answers should be in the form of True or False..
        
        Do not provide any conversation, only the boolean True/False as your answer.
        """
        #print()
        #print(len(prompt))
        
        response = prompt_and_response(oll_client, model, prompt, quiet=quiet, options=options)
        if response is not None:
            #print(type(response))
            j_response = {
                'src_node': node_pair['src_node'],
                'pair_node': node_pair['pair_node'],
                'is_connected': response
            }
            part1_responses.append(j_response)
        else:
            if not quiet:
                print(f'Response for {node_pair['src_node']['_id']} -> {node_pair['pair_node']['_id']} is None.')
        
    
    prompts_total = datetime.now() - prompts_start
    if not quiet:
        print(f'***** Prompts took a total of: {prompts_total} *****')

    return part1_responses

In [16]:
def part2_prompt(part1_responses, oll_client, model='llama3.3:70b', quiet=True, options={'temperature': 0.8}):
    part2_responses = []
    max_pairs_listed = 10
    prompts_start = datetime.now()
    pairings = [
        {
            'src_node': edge['src_node'], 
            'pair_node': edge['pair_node']
        } for edge in part1_responses if edge['is_connected'] #get_graph_edges(db, aql, graph_name, include_node_docs=True) if edge['_from_node'] is not None and edge['_to_node'] is not None
    ]
    
    for node_pair in tqdm(pairings, desc='Prompt 2', leave=False):
        #print(json.dumps(node_pair, indent=4))
        #print(node_pair["src_node"]["_id"])
        
        prompt = f"""Act as a data analyzer. Consider the following data for a graph network, as a source node and a destination node:
        {json.dumps(node_pair)}
        
        What relationship or edge type is most likely between these two nodes? Justify with field-level analysis.
        The edge should be one of the following types: 
            - LEADS_TO : a sequential relationship where node A directly precedes node B
            - REFERENCES : a relationship where node B provides insights for the production or completion of node A
            - CONTAINS : a realtionship where node A contains an instance of node B or node B is a part of node A, e.g., a process CONTAINS a step because a step is part of the process
            - PRODUCES : node A produces node B, e.g., a planning step produces a planning document
        Answers should be in the form of 1 single JSON object (not a list) such as below, with 0 or 1 entry per pair:    
        '{{"src_attr": "<src_node_attribute>", "dest_attr": "<pair_node_attribute>", "explanation": "<1 sentence explanation of why the nodes are connected>"}}'.
        For Example:
        {{"src_attr": "attribute1", "dest_attr": "attribute3", "type": "REFERENCES", "explanation": "destination node's attribute3 field is related to the source node's attribute 1 field"}}
    
        If there are no obvious explicit connections, set the explanation to "NO CONNECTION".
        
        Do not provide any conversation, only the JSON object as your answer.
        """
        #print()
        #print(len(prompt))
        
        response = prompt_and_response(oll_client, model, prompt, quiet=quiet, options=options)
        if response is not None:
            #print(response)
            if isinstance(response, list):
                for r in response:
                    r = json.loads(r.replace('```json', '').replace('```', '')) 
                    r['_from'] = node_pair['src_node']['_id']
                    r['_to'] = node_pair['pair_node']['_id']
                    r['src_node'] = node_pair['src_node']
                    r['pair_node'] = node_pair['pair_node']
                    part2_responses.append(r)
            else:
                response = json.loads(response.replace('```json', '').replace('```', '')) 
                response['_from'] = node_pair['src_node']['_id']
                response['_to'] = node_pair['pair_node']['_id']
                response['src_node'] = node_pair['src_node']
                response['pair_node'] = node_pair['pair_node']
                part2_responses.append(response)
        else:
            if not quiet:
                print(f'Response for {node_pair['src_node']['_id']} -> {node_pair['pair_node']['_id']} is None.')
        
    
    prompts_total = datetime.now() - prompts_start
    if not quiet:
        print(f'***** Prompts took a total of: {prompts_total} *****')

    return part2_responses

In [17]:
def part3_prompt(part2_responses, oll_client, model='gemma3:27b-it-qat', quiet=True, options={'temperature': 0.8}):
    part3_responses = []
    max_pairs_listed = 10
    prompts_start = datetime.now()
    pairings = [
        {
            'src_node': edge['src_node'], 
            'pair_node': edge['pair_node']
        } for edge in part2_responses #get_graph_edges(db, aql, graph_name, include_node_docs=True) if edge['_from_node'] is not None and edge['_to_node'] is not None
    ]
    
    for i in tqdm(range(0, len(pairings)), desc='Prompt 3', leave=False):
        node_pair = pairings[i]
        #print(node_pair)
        #print(node_pair["src_node"]["_id"])
        
        prompt = f"""
        Act as a data analyzer. Consider the following data for a graph network, as a source node and a destination node:
        {json.dumps(node_pair)}
        
        Rate the strength and accuracy of a potential connection from 1-10.
        Answers should be in the form of an integer between 1 and 10.
        
        Example output: "9"
        Think through this problem and then provide me ONLY the conclusion. Do not show your reasoning process in the final answer.    
        """
        #print()
        #print(len(prompt))
        
        response = prompt_and_response(oll_client, model, prompt, quiet=quiet, options=options)
        if response is not None:
            #print(response)
            j_response = {
                'src_node': node_pair['src_node'],
                'pair_node': node_pair['pair_node'],
                '_from': node_pair['src_node']['_id'],
                '_to': node_pair['pair_node']['_id'],
                'type': part2_responses[i]['type'],
                'explanation': part2_responses[i]['explanation'],
                'src_attr': part2_responses[i]['src_attr'],
                'dest_attr': part2_responses[i]['dest_attr'],
                'conn_strength': int(response),
                'status': 'success:p3'
            }
            part3_responses.append(j_response)
        else:
            if not quiet:
                print(f'Response for {node_pair['src_node']['_id']} -> {node_pair['pair_node']['_id']} is None.')
            """j_response = {
                'src_node': p2_response['src_node'],
                'pair_node': p2_response['pair_node'],
                '_from': p2_response['src_node']['_id'],
                '_to': p2_response['pair_node']['_id'],
                'type': p2_response['type'],
                'explanation': p2_response['explanation'],
                'src_attr': p2_response['src_attr'],
                'dest_attr': p2_response['dest_attr'],
                'conn_strength': int(response),
                'status': f'ERROR: {}'
            }"""
        
    
    prompts_total = datetime.now() - prompts_start
    if not quiet:
        print(f'***** Prompts took a total of: {prompts_total} *****')

    return part3_responses

In [18]:
def append_json_to_file(new_entry, file_path, write_lock):
    with write_lock:
        # Check if file exists
        with open(file_path, 'a') as file:
            file.write(json.dumps(new_entry, default=str) + '\n')

## Variables and Setup

In [19]:
############ USER VARIABLES ############
ar_host = 'http://localhost:8529'
db_name = 'Process_Test'
username = 'root'
graph_name = 'Test_Process'

oll_host = 'http://10.10.80.99:4001'
GEMMA3_27B_MODEL = 'gemma3:27b-it-qat'
GEMMA3_12B_MODEL = 'gemma3:12b-it-qat'
LLAMA3_3_70B_MODEL = 'llama3.3:70b'
MAGISTRAL_24B_MODEL = 'magistral:24b'
CODELLAMA_70B_MODEL = 'codellama:70b'
DEEPSEEK_R1_70B_MODEL = 'deepseek-r1:70b'
#oll_sys_settings = "You are a specialized software for assisting in creating graph networks. You will receive node data and provide edge definitions as output in JSON format. Do not engage in any conversation."

n_samples = 10
pairs_per_sample = 40
src_filter_str = ''
pairs_filter_str = ''
src_collections = []
pairs_collections = [] #['TTPArtifact']
exclude_src_collections = ['Process', 'PlanningStep', 'DevelopmentStep', 'ExecutionStep', 'TTPArtifact']
exclude_pairs_collections = ['Process', 'PlanningStep', 'DevelopmentStep', 'ExecutionStep', 'TTPArtifact']
########## END USER VARIABLES ##########

In [20]:
password = getpass.getpass(f'Please enter the password for user {username} for access to the {db_name} database: ')

Please enter the password for user root for access to the Process_Test database:  ········


In [21]:
ar_client = connect_to_arango_client(ar_host)
db = connect_to_arango_db(ar_client, db_name, username, password)
aql = db.aql
oll_client = Client(host=oll_host)

2025-10-22 09:45:04 -- Successfully connected to database Process_Test running version 3.12.5-2.


In [22]:
pairings = get_sample_pairs_from_graph(db, aql, graph_name, 
                                       n_samples=n_samples, 
                                       pairs_per_sample=pairs_per_sample,
                                       src_filter_str=src_filter_str,
                                       pairs_filter_str=pairs_filter_str,
                                       src_collections=src_collections,
                                       pairs_collections=pairs_collections,
                                       exclude_src_collections=exclude_src_collections,
                                       exclude_pairs_collections=exclude_pairs_collections
                                      )

pairing_ids = [{'src_id': pair['src_node']['_id'], 'pair_ids': [pair_node['_id'] for pair_node in pair['pair_nodes']]} for pair in pairings]
total_pairs = sum([len(p["pair_ids"]) for p in pairing_ids])
print(f'Total combinations: {total_pairs}')
print(json.dumps(pairing_ids, indent=4))

Total combinations: 2640
[
    {
        "src_id": "APTProfileArtifact/obap_apt_001",
        "pair_ids": [
            "StorylineArtifact/obap_storyline_001",
            "RedTeamDocArtifact/obap_red_doc_001",
            "ProcessLogArtifact/4390",
            "SprintLogArtifact/2024",
            "RobotLogArtifact/2133",
            "OrchestrationPlanArtifact/obap_orchestration_seq_001",
            "ProcessLogArtifact/4389",
            "RobotLogArtifact/2130",
            "RobotLogArtifact/2128",
            "LearningObjectivesArtifact/obap_dlo_001",
            "ConfluenceDocArtifact/obap_confluence_page",
            "RobotLogArtifact/2132",
            "ProcessLogArtifact/4396",
            "SprintLogArtifact/2025",
            "FeedbackImplementationArtifact/obap_feedback_impl_001",
            "VMDeploymentArtifact/obap_deployment_001",
            "ProcessLogArtifact/4388",
            "DeadRangeValidationArtifact/obap_dead_range_001",
            "ExecutionCertificationArtif

## Run Experiment Table

In [54]:
def gen_consistency_score(data):
    return 100 / (data['exact_match_range'] + data['partial_match_range'] + data['inc_kv_counts_range'] + data['missed_range'] + 1)
    
def gen_model_overall_score(data):
    return ( (data['avg_exact_match']/(data['exact_match_range'] + 1)) + ((data['avg_partial_correct'] - (data['avg_avg_inc_kv_counts']*10)) / (data['partial_correct_range']) + 1) ) / data['total_expected']
    

In [71]:
pd.options.display.max_columns = None
grid_results_df = pd.read_json('grid_results_102925_01.json', lines=True)
print(len(grid_results_df))
grid_results_df_sorted_avgc = grid_results_df.sort_values(by=['avg_avg_inc_kv_counts', 'avg_exact_match', 'avg_partial_correct', 'avg_runtime'], ascending=[True, False, False, True])

grid_results_df_sorted_avgc

52


,model1,model2,temp1,temp2,iterations,total_expected,avg_exact_match,max_exact_match,min_exact_match,exact_match_range,avg_partial_correct,max_partial_correct,min_partial_correct,partial_correct_range,avg_incorrect,max_incorrect,min_incorrect,incorrect_range,avg_missed,max_missed,min_missed,missed_range,avg_avg_inc_kv_counts,max_inc_kv_counts,min_inc_kv_counts,inc_kv_counts_range,avg_runtime,max_runtime,min_runtime
21,gpt-oss:120b,llama3.3:70b,0.8,0.5,"[{'total_expected': 107, 'total_exact_match': ...",107,98.2,99,97,2,7.8,9,7,2,1.0,1,1,0,0.0,0,0,0,0.091589,2,0,2,1067.176583,1360.888094,785.717360
7,gpt-oss:120b,llama3.3:70b,0.2,1.0,"[{'total_expected': 107, 'total_exact_match': ...",107,98.0,99,97,2,8.0,9,7,2,1.0,1,1,0,0.0,0,0,0,0.093458,2,0,2,1082.975957,1270.013120,865.711584
38,gemma3:27b-it-qat,llama3.3:70b,0.2,1.0,"[{'total_expected': 107, 'total_exact_match': ...",107,97.8,98,97,1,8.2,9,8,1,1.0,1,1,0,0.0,0,0,0,0.095327,2,0,2,580.918283,832.393458,394.555745
13,gpt-oss:120b,llama3.3:70b,0.5,0.5,"[{'total_expected': 107, 'total_exact_match': ...",107,97.8,98,97,1,8.2,9,8,1,1.0,1,1,0,0.0,0,0,0,0.095327,2,0,2,1123.911848,1334.686665,807.669128
6,gpt-oss:120b,llama3.3:70b,0.2,0.8,"[{'total_expected': 107, 'total_exact_match': ...",107,97.8,98,97,1,8.2,9,8,1,1.0,1,1,0,0.0,0,0,0,0.095327,2,0,2,1230.720032,1319.612471,1137.171237
46,gemma3:27b-it-qat,llama3.3:70b,0.5,1.0,"[{'total_expected': 107, 'total_exact_match': ...",107,97.6,99,96,3,8.4,10,7,3,1.0,1,1,0,0.0,0,0,0,0.097196,2,0,2,465.095439,596.584615,403.328087
28,gpt-oss:120b,llama3.3:70b,1.0,0.5,"[{'total_expected': 107, 'total_exact_match': ...",107,97.6,99,96,3,8.4,10,7,3,1.0,1,1,0,0.0,0,0,0,0.097196,2,0,2,1025.875565,1116.193441,955.260882
12,gpt-oss:120b,llama3.3:70b,0.5,0.2,"[{'total_expected': 107, 'total_exact_match': ...",107,97.6,99,97,2,8.4,9,7,2,1.0,1,1,0,0.0,0,0,0,0.097196,2,0,2,1192.209603,1368.962952,898.869082
22,gpt-oss:120b,llama3.3:70b,0.8,0.8,"[{'total_expected': 107, 'total_exact_match': ...",107,97.6,98,97,1,8.4,9,8,1,1.0,1,1,0,0.0,0,0,0,0.097196,2,0,2,1238.972950,1387.843138,1049.611440
51,gemma3:27b-it-qat,llama3.3:70b,0.8,0.2,"[{'total_expected': 107, 'total_exact_match': ...",107,97.4,98,97,1,8.6,9,8,1,1.0,1,1,0,0.0,0,0,0,0.099065,2,0,2,652.391110,782.246993,444.142475


In [63]:
grid_results_df['model_score'] = gen_model_overall_score(grid_results_df)
grid_results_df_sorted_score = grid_results_df.sort_values(by='model_score', ascending=False)
grid_results_df_sorted_score

,model1,model2,temp1,temp2,iterations,total_expected,avg_exact_match,max_exact_match,min_exact_match,exact_match_range,avg_partial_correct,max_partial_correct,min_partial_correct,partial_correct_range,avg_incorrect,max_incorrect,min_incorrect,incorrect_range,avg_missed,max_missed,min_missed,missed_range,avg_avg_inc_kv_counts,max_inc_kv_counts,min_inc_kv_counts,inc_kv_counts_range,avg_runtime,max_runtime,min_runtime,model_score
28,gemma3:27b-it-qat,llama3.3:70b,1.0,0.5,"[{'total_expected': 107, 'total_exact_match': ...",107,96.4,98,94,4,9.6,12,8,4,1.0,1,1,0,0.0,0,0,0,0.108411,2,0,2,0:05:31.939283,0:05:34.608645,0:05:29.419585,0.002171
22,gemma3:27b-it-qat,llama3.3:70b,0.8,1.0,"[{'total_expected': 107, 'total_exact_match': ...",107,95.8,98,94,4,10.2,12,8,4,1.0,1,1,0,0.0,0,0,0,0.114019,2,0,2,0:05:30.993555,0:05:33.003883,0:05:27.306079,0.001949
19,gemma3:27b-it-qat,llama3.3:70b,0.8,0.2,"[{'total_expected': 107, 'total_exact_match': ...",107,96.4,98,95,3,9.6,11,8,3,1.0,1,1,0,0.0,0,0,0,0.108411,2,0,2,0:05:32.841466,0:05:38.293548,0:05:29.168391,0.001794
7,gemma3:27b-it-qat,llama3.3:70b,0.2,1.0,"[{'total_expected': 107, 'total_exact_match': ...",107,94.6,96,92,4,11.4,14,10,4,1.0,1,1,0,0.0,0,0,0,0.125234,2,0,2,0:05:32.358153,0:05:35.941668,0:05:29.402114,0.001623
29,gemma3:27b-it-qat,llama3.3:70b,1.0,0.8,"[{'total_expected': 107, 'total_exact_match': ...",107,95.6,97,94,3,10.4,12,9,3,1.0,1,1,0,0.0,0,0,0,0.115888,2,0,2,0:05:30.431961,0:05:33.402913,0:05:27.543133,0.001598
5,gemma3:27b-it-qat,llama3.3:70b,0.2,0.5,"[{'total_expected': 107, 'total_exact_match': ...",107,95.6,97,94,3,10.4,12,9,3,1.0,1,1,0,0.0,0,0,0,0.115888,2,0,2,0:05:34.750004,0:05:37.476138,0:05:31.233308,0.001598
13,gemma3:27b-it-qat,llama3.3:70b,0.5,0.5,"[{'total_expected': 107, 'total_exact_match': ...",107,96.0,97,95,2,10.0,11,9,2,1.0,1,1,0,0.0,0,0,0,0.112150,2,0,2,0:05:32.421534,0:05:39.342626,0:05:29.670604,0.001458
21,gemma3:27b-it-qat,llama3.3:70b,0.8,0.8,"[{'total_expected': 107, 'total_exact_match': ...",107,96.0,97,95,2,10.0,11,9,2,1.0,1,1,0,0.0,0,0,0,0.112150,2,0,2,0:05:30.293560,0:05:36.063808,0:05:27.088733,0.001458
27,gemma3:27b-it-qat,llama3.3:70b,1.0,0.2,"[{'total_expected': 107, 'total_exact_match': ...",107,96.0,97,95,2,10.0,11,9,2,1.0,1,1,0,0.0,0,0,0,0.112150,2,0,2,0:05:33.971406,0:05:42.841781,0:05:29.339271,0.001458
18,gemma3:27b-it-qat,gemma3:27b-it-qat,0.8,1.0,"[{'total_expected': 107, 'total_exact_match': ...",107,89.4,92,87,5,15.2,18,12,6,2.4,3,2,1,0.0,0,0,0,0.190654,3,0,3,0:04:12.250873,0:04:14.887393,0:04:09.466573,0.001392
